# 01 — Download + Camada RAW

**Objetivo:** baixar todos os `CAGEDMOV` de 2021–2025, extrair os `.txt` e gravar a **camada raw** (parquet fiel à fonte, sem filtro). Idempotente.

In [ ]:
import sys, pathlib
# garante que o pacote caged_etl (em ../) seja importável a partir do notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from caged_etl import setup_logging, get_settings
setup_logging()
settings = get_settings()
print('Municípios:', settings.municipios, '| Anos:', settings.ano_inicio, '-', settings.ano_fim)

## 1. Download de toda a janela (2021–2025)
Pode demorar — cada mês tem dezenas de MB. Já baixados são pulados.

In [ ]:
from caged_etl import ftp_client
baixados = ftp_client.download_range()
print(f'{len(baixados)} arquivos disponíveis em downloads/')

## 2. Extração de todos os .7z

In [ ]:
from caged_etl import extract
txts = extract.extract_all()
print(f'{len(txts)} arquivos .txt extraídos')

## 3. Construção da camada RAW (parquet por competência)

In [ ]:
from caged_etl import raw
parquets = raw.build_raw_layer()
print(f'{len(parquets)} parquets raw gravados em {settings.raw_dir}')

## 4. Sanidade: linhas por competência

In [ ]:
import polars as pl
lf = pl.scan_parquet(sorted(settings.raw_dir.glob('caged_mov_*.parquet')))
por_comp = (lf.group_by('competenciamov').len()
              .sort('competenciamov').collect())
print('total de linhas (Brasil, sem filtro):', por_comp['len'].sum())
por_comp